# E3 · A colleague discovers that wines are a one-dimensional problem

**Type**: Find the bug · **Level**: ●●○ independent · **Time**: about 35 minutes · **core path ★**

## Objective

Find out why a PCA on the wines gives 99.8% of the variance to the first component ("the dataset is almost one-dimensional") and a classifier on that single component reaches 0.70: the PCA was done **on the raw data**, where `proline` (scale of hundreds) decides alone. Fix it by
standardizing and verify how false the conclusion was.

## Dataset

`load_wine`: 178 wines, 13 measurements with very different scales (`proline` up to 1680, `hue` around 1), 3 classes. Validation `StratifiedKFold(5, shuffle=True, random_state=0)`.

## Educational purpose

Recognize the most common mistake in the use of PCA (not standardizing) and its signature: a first component with 99+% of the variance and a loading almost entirely on a single column. It is not a discovery about the phenomenon: it is an effect of the units of measure.

## Task

1. Run the colleague's cell: `raw_pc1_ratio` and `raw_1_acc` (logistic on the first component alone, in cross-validation).
2. Save `pc1_column` (name of the column with the largest loading in PC1) and `proline_variance_share` (variance of `proline` divided by the sum of the variances of all the columns).
3. Fix it: pipeline `StandardScaler` → `PCA(k)` → logistic. Save `scaled_pc1_ratio` (variance explained by PC1 after scaling), `scaled_1_acc` (k = 1) and `scaled_2_acc` (k = 2).
4. Save `scaled_k_for_90`: components needed for 90% of the variance on the standardized data (against 1 in the raw data).

## Setup

Run this cell without changing it.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.datasets import load_wine
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

X, y = load_wine(return_X_y=True, as_frame=True)
cv = StratifiedKFold(5, shuffle=True, random_state=0)

## Your work

### The colleague's code

In [ ]:
pca = PCA().fit(X)
raw_pc1_ratio = pca.explained_variance_ratio_[0]
raw_1_acc = cross_val_score(make_pipeline(PCA(1), LogisticRegression(max_iter=3000)), X, y, cv=cv).mean()
print(f"variance explained by PC1: {raw_pc1_ratio:.3f} -> 'the problem is one-dimensional'. Accuracy with 1 component: {raw_1_acc:.3f}")

### Your diagnosis and the fix

In [ ]:
pc1_column = None
proline_variance_share = None

In [ ]:
scaled_pc1_ratio = None
scaled_1_acc = None
scaled_2_acc = None
scaled_k_for_90 = None

## Check

Run the cell when you think you are done. If a check fails, the message tells you what is wrong. If it fails a second time, a hint appears under the message.

In [ ]:
with verify(hint='`X.var().idxmax()` and `X.var()["proline"] / X.var().sum()`. Column of PC1: `X.columns[np.abs(pca.components_[0]).argmax()]`. Components for 90%: `np.searchsorted(np.cumsum(PCA().fit(StandardScaler().fit_transform(X)).explained_variance_ratio_), 0.9) + 1`.'):
    check(raw_pc1_ratio > 0.99 and raw_1_acc < 0.75, "the colleague: PC1 with over 99% of the variance and accuracy below 0.75", "don't modify the colleague's code")
    check(pc1_column == "proline" and proline_variance_share is not None and proline_variance_share > 0.99, "PC1 is the column `proline`, which has over 99% of the total variance", "X.var() and components_[0]")
    check(scaled_pc1_ratio is not None and 0.3 < scaled_pc1_ratio < 0.4 and scaled_k_for_90 == 8, "after scaling PC1 explains 36%, and 8 components are needed for 90%", "StandardScaler before PCA")
    check(scaled_2_acc is not None and scaled_2_acc > 0.95 and scaled_2_acc > raw_1_acc + 0.2, "with two components of the standardized data the accuracy is over 0.95 (against 0.70)", "make_pipeline(StandardScaler(), PCA(2), LogisticRegression())")
    check(scaled_1_acc > raw_1_acc + 0.1, "even with a single component scaling improves by over 0.1", "compare scaled_1_acc and raw_1_acc")
done()

---
The commented solution is in `solutions/E_find_the_bug/E3_pca_on_the_raw_wine_data.ipynb`. Open it after you pass the check, or after you have really tried.